# Exercise D1

### How Deep Do Immune Cells Penetrate? (Spatial Infiltration Profiles)

An immune-excluded tumour has T cells piling up at the tumour edge but not getting in. An inflamed
tumour has them distributed throughout the core. Pathologists reason about this distinction every
time they look at a slide; we just want to make it quantitative and reproducible across all 15 patients.


### Background: 
For each cell, define its border distance as the distance to the nearest cell of the
opposite tissue type (tumour cell nearest to stroma, or stroma cell nearest to tumour). Positive
values = deeper inside the tissue compartment.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree
from sklearn.preprocessing import normalize
import pathlib as pth

# Selecting & Importing df

In [2]:
path_to_files: pth.Path = pth.Path("files_with_cell_types")


for i, file_path in enumerate(path_to_files.iterdir()):
    print(f"[{i}]:\t{file_path}")

[0]:	files_with_cell_types/LUNG-NSCLC2-0613-FIXT-01-IF1-01_#_cells_properties_#_53ec91fc9c63f3088eb688a53912ac53.tsv.gz.csv
[1]:	files_with_cell_types/LUNG-NSCLC2-0664-FIXT-01-IF1-01_#_cells_properties_#_e2995fa6cfa5cfdbd51cc83e8e465177.tsv.gz.csv
[2]:	files_with_cell_types/LUNG-NSCLC2-0695-FIXT-01-IF1-01_#_cells_properties_#_8f2bfb1ba17fa457f1fb5b8650a45cb1.tsv.gz.csv
[3]:	files_with_cell_types/LUNG-NSCLC2-0696-FIXT-01-IF1-01_#_cells_properties_#_9f8d9e704403e21802101969e9adef0d.tsv.gz.csv
[4]:	files_with_cell_types/LUNG-NSCLC2-0657-FIXT-01-IF1-01_#_cells_properties_#_c1cca8eabb109ca29cb7c2a4530fd76e.tsv.gz.csv
[5]:	files_with_cell_types/LUNG-NSCLC2-0587-FIXT-01-IF1-01_#_cells_properties_#_fb439474f2166fe87b6d3087643b2af3.tsv.gz.csv
[6]:	files_with_cell_types/LUNG-NSCLC2-0591-FIXT-01-IF1-01_#_cells_properties_#_b17f2afaad57821d4888ff6ba7f4b179.tsv.gz.csv
[7]:	files_with_cell_types/LUNG-NSCLC2-0659-FIXT-01-IF1-01_#_cells_properties_#_1a6b8562910080e4a4e518975b98dad9.tsv.gz.csv
[8]:	fil

In [3]:
df = pd.read_csv("files_with_cell_types/LUNG-NSCLC2-0613-FIXT-01-IF1-01_#_cells_properties_#_53ec91fc9c63f3088eb688a53912ac53.tsv.gz.csv", index_col=[0])

df

,cell.ID,nucleus.x,nucleus.y,CD15.score,CK.score,CD3.score,CD11c.score,CD20.score,CD163.score,CD15.score.normalized,CK.score.normalized,CD3.score.normalized,CD11c.score.normalized,CD20.score.normalized,CD163.score.normalized,tissue.type,phenotype,in.ROI.next_to_tumor_tissue,in.ROI.tumor_tissue,cell.type
0,140,7270.2,34129.1,1.646708,6.967399,0.000000,0.773145,0.025275,5.087766,0.1647,1.4111,0.0000,0.0751,0.0085,0.3381,stroma,CD15-CK+CD3-CD11c-CD20-CD163-,False,False,CD15-Tumor
1,141,6773.0,34155.4,1.611525,4.887243,0.000000,0.406094,0.000000,88.199103,0.1612,0.9898,0.0000,0.0394,0.0000,5.8604,stroma,CD15-CK-CD3-CD11c-CD20-CD163+,False,True,Macrophage
2,142,6995.3,34304.3,1.482387,4.199115,1.202758,0.387839,0.035550,2.137255,0.1482,0.8504,0.0932,0.0377,0.0120,0.1420,stroma,CD15-CK-CD3-CD11c-CD20-CD163-,False,True,other
3,143,6954.6,33921.2,0.616335,3.908355,1.478800,0.406528,0.012177,4.902202,0.0616,0.7915,0.1146,0.0395,0.0041,0.3257,stroma,CD15-CK-CD3-CD11c-CD20-CD163-,False,False,other
4,144,7109.9,34277.5,0.786399,2.337006,27.430016,0.444674,0.032040,4.650998,0.0786,0.4733,2.1264,0.0432,0.0108,0.3090,stroma,CD15-CK-CD3+CD11c-CD20-CD163-,False,True,Tcell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202423,1218012,6675.8,48899.5,0.866750,11.760290,33.399130,0.672551,1.340717,4.040546,0.0867,2.3818,2.5891,0.0653,0.4529,0.2685,tumor,CD15-CK+CD3+CD11c-CD20-CD163-,False,True,Tcell
1202424,1218013,6234.7,48801.8,1.051189,4.889122,83.125742,0.983521,0.393322,7.960153,0.1051,0.9902,6.4439,0.0955,0.1329,0.5289,stroma,CD15-CK-CD3+CD11c-CD20-CD163-,False,True,Tcell
1202425,1218014,6364.7,48716.9,0.866861,7.639275,16.710003,0.704519,0.301394,3.946967,0.0867,1.5472,1.2953,0.0684,0.1018,0.2623,tumor,CD15-CK+CD3+CD11c-CD20-CD163-,False,True,Tcell
1202426,1218015,6596.9,48696.1,1.022252,1.939428,11.016228,0.597027,2.770626,22.985049,0.1022,0.3928,0.8540,0.0580,0.9360,1.5272,stroma,CD15-CK-CD3-CD11c-CD20-CD163+,False,True,Macrophage


# Task 1 

Build a cKDTree on stroma cells and query every tumour cell, and vice versa. Call the result border_dist. Add it as a new column to the dataframe.

In [23]:
def calculate_distances_using(df_based: pd.DataFrame, df_query: pd.DataFrame, neigh_num=10) -> np.ndarray:

    tree = cKDTree(df_based[["nucleus.x", "nucleus.y"]].to_numpy())

    distances, _ = tree.query(df_query[["nucleus.x", "nucleus.y"]].to_numpy(), k=neigh_num + 1)
    knn_distances = distances[:, 1:]

    return knn_distances

In [24]:
df["tissue.type"].value_counts()

tissue.type
stroma    701034
tumor     501394
Name: count, dtype: int64

In [25]:
df_stroma = df[df["tissue.type"] == "stroma"]

df_tumor = df[df["tissue.type"] == "tumor"]

In [27]:
df_stroma["border_dist"] = calculate_distances_using(
    df_stroma, df_tumor
)

df_tumor["border_dist"] = calculate_distances_using(
    df_tumor, df_stroma
)

ValueError: Length of values (501394) does not match length of index (701034)